# Company A Genre & Language Classification

## Data Source
- **Input**: company_a.csv (checkout frequency data)
- **Data Type**: Checkout transactions (each "Number" represents checkout count)
- **Processing**: Detects language, assigns genres, generates enriched output with analytics

See [DATA_SEMANTICS.md](../../Data/DATA_SEMANTICS.md) for details on data interpretation across all companies.

In [5]:
import os
import json
import re
import unicodedata
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
from functools import lru_cache
import concurrent.futures

import pandas as pd
import langid
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **k: x

import warnings
warnings.filterwarnings("ignore", category=Warning)

# =========================================================
# CONFIG
# =========================================================
COMPANY_A_FILE = "company_a.csv"
OUTPUT_COMPANY_A = "company_a_genres_output.csv"
TOP_K_GENRES = 3
OPENLIBRARY_CACHE_FILE = "openlibrary_cache.json"

# =========================================================
# LANGUAGE DETECTION
# =========================================================
LANGUAGE_NAMES = {
    "en": "English",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "hy": "Armenian",
    "tr": "Turkish",
    "ar": "Arabic",
    "zh": "Chinese",
    "ja": "Japanese",
    "ko": "Korean",
}

EN_HINT_WORDS = frozenset({"the", "a", "an", "of", "and", "to", "in", "for", "with", "on"})

# =========================================================
# GENRE MAPPING
# =========================================================
GENRES = [
    "Fantasy",
    "Science Fiction",
    "Romance",
    "Mystery",
    "Thriller",
    "Historical Fiction",
    "Nonfiction",
    "Biography",
    "Young Adult",
    "Horror",
]

GENRE_KEYWORDS = {
    "Fantasy": frozenset(["fantasy", "magic", "dragon", "myth", "middle earth"]),
    "Science Fiction": frozenset(["science fiction", "sci-fi", "space", "alien", "dystop"]),
    "Romance": frozenset(["romance", "love"]),
    "Mystery": frozenset(["mystery", "detective", "crime"]),
    "Thriller": frozenset(["thriller", "suspense"]),
    "Horror": frozenset(["horror", "ghost", "haunted"]),
    "Biography": frozenset(["biography", "autobiography", "memoir"]),
    "Nonfiction": frozenset(["nonfiction", "history", "business", "psychology", "self-help"]),
    "Historical Fiction": frozenset(["historical fiction"]),
    "Young Adult": frozenset(["young adult", "ya"]),
}

WHITESPACE_REGEX = re.compile(r"\s+")


@lru_cache(maxsize=1024)
def detect_language(title: str, min_confidence_latin: float = 0.35) -> Tuple[str, str, float]:
    text = (title or "").strip()
    if not text:
        return "unknown", "Unknown", 0.0

    # quick-script heuristics
    for ch in text:
        cp = ord(ch)
        if 0x0530 <= cp <= 0x058F:
            return "hy", "Armenian", 1.0
        if 0x0400 <= cp <= 0x04FF:
            return "cyr", "Cyrillic", 1.0
        if 0x0600 <= cp <= 0x06FF:
            return "ar", "Arabic", 1.0
        if 0x0590 <= cp <= 0x05FF:
            return "he", "Hebrew", 1.0
        if 0x0370 <= cp <= 0x03FF:
            return "el", "Greek", 1.0
        if 0x4E00 <= cp <= 0x9FFF:
            return "cjk", "CJK", 1.0

    normalized = unicodedata.normalize("NFKD", text)
    normalized = "".join(c for c in normalized if not unicodedata.combining(c))

    words = {w.lower() for w in normalized.replace("'", " ").split()}
    if words & EN_HINT_WORDS:
        return "en", "English", 0.99

    code, conf = langid.classify(normalized)
    if conf >= min_confidence_latin:
        return code, LANGUAGE_NAMES.get(code, code), float(conf)

    return "latin", "Latin (Unknown language)", float(conf)


# =========================================================
# OPEN LIBRARY (cache + retries)
# =========================================================
OPENLIBRARY_CACHE: Dict[str, List[str]] = {}


def _load_cache():
    try:
        if os.path.exists(OPENLIBRARY_CACHE_FILE):
            with open(OPENLIBRARY_CACHE_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, dict):
                    OPENLIBRARY_CACHE.update(data)
    except Exception:
        pass


def _save_cache():
    try:
        with open(OPENLIBRARY_CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(OPENLIBRARY_CACHE, f, ensure_ascii=False)
    except Exception:
        pass


_session: Optional[requests.Session] = None

def _session_with_retries():
    global _session
    if _session is None:
        s = requests.Session()
        retries = Retry(total=3, backoff_factor=0.6, status_forcelist=(500,502,503,504))
        s.mount("https://", HTTPAdapter(max_retries=retries))
        _session = s
    return _session


_load_cache()


def clean_title_for_lookup(title: str) -> str:
    """Remove Armenian/language suffixes and formatting"""
    title = (title or "").strip()
    # Remove Armenian suffix ", հատ" (means "piece/copy")
    title = title.replace(", հատ", "").strip()
    # Remove common suffixes
    for suffix in [" - հատ", " հատ", " (հատ)"]:
        if title.endswith(suffix):
            title = title[:-len(suffix)].strip()
    return title


def openlibrary_get_subjects(title: str) -> List[str]:
    """Disk-backed cached lookup with a shared session and retries."""
    title = (title or "").strip()
    if not title:
        return []
    if title in OPENLIBRARY_CACHE:
        return OPENLIBRARY_CACHE[title]

    # Clean title for better matching
    clean_title = clean_title_for_lookup(title)
    cache_key = clean_title if clean_title != title else title
    
    if cache_key in OPENLIBRARY_CACHE:
        return OPENLIBRARY_CACHE[cache_key]

    session = _session_with_retries()
    try:
        r = session.get("https://openlibrary.org/search.json", params={"title": clean_title}, timeout=8)
        r.raise_for_status()
        data = r.json()
        docs = data.get("docs", [])
        if docs and docs[0].get("subject"):
            subjects = docs[0]["subject"]
            OPENLIBRARY_CACHE[cache_key] = subjects
            return subjects
        if docs:
            work_key = docs[0].get("key")
            if work_key:
                w = session.get(f"https://openlibrary.org{work_key}.json", timeout=8)
                w.raise_for_status()
                subjects = w.json().get("subjects", []) or []
                OPENLIBRARY_CACHE[cache_key] = subjects
                return subjects
    except Exception:
        OPENLIBRARY_CACHE[cache_key] = []
        return []
    OPENLIBRARY_CACHE[cache_key] = []
    return []


def normalize(s: str) -> str:
    return WHITESPACE_REGEX.sub(" ", (s or "").lower().strip())


def map_subjects_to_genres(subjects: List[str], top_k: int) -> List[str]:
    if not subjects:
        return []
    text = " | ".join(normalize(x) for x in subjects)
    found = []
    for genre, keys in GENRE_KEYWORDS.items():
        if any(k in text for k in keys):
            found.append(genre)
            if len(found) >= top_k:
                break
    return found[:top_k]


# =========================================================
# AI FALLBACK (lazy + batched)
# =========================================================
_classifier = None
_classifier_model = "valhalla/distilbart-mnli-12-1"


def get_classifier():
    global _classifier
    if _classifier is None:
        try:
            import torch
            from transformers import pipeline, AutoConfig
            device = 0 if torch.cuda.is_available() else -1
            cfg = AutoConfig.from_pretrained(_classifier_model)
            cfg.tie_word_embeddings = False
            _classifier = pipeline("zero-shot-classification", model=_classifier_model, config=cfg, device=device)
        except Exception:
            _classifier = None
    return _classifier


def simple_genre_detection(title: str) -> List[str]:
    """Simple keyword-based genre detection as last resort fallback"""
    title_lower = (title or "").lower()
    detected = []
    
    # Simple keyword matching
    if any(word in title_lower for word in ["97", "1984", "science", "future", "robot", "space"]):
        detected.append("Science Fiction")
    if any(word in title_lower for word in ["mystery", "detective", "crime", "murder", "secret"]):
        detected.append("Mystery")
    if any(word in title_lower for word in ["love", "romance", "passion", "heart"]):
        detected.append("Romance")
    if any(word in title_lower for word in ["young", "teen", "youth", "adolescent"]):
        detected.append("Young Adult")
    if any(word in title_lower for word in ["history", "historical", "past", "war", "ancient"]):
        detected.append("Historical Fiction")
    if any(word in title_lower for word in ["bio", "auto", "memoir", "life story"]):
        detected.append("Biography")
    if any(word in title_lower for word in ["fantasy", "magic", "dragon", "myth", "quest"]):
        detected.append("Fantasy")
    if any(word in title_lower for word in ["horror", "ghost", "haunted", "fear", "terror"]):
        detected.append("Horror")
    if any(word in title_lower for word in ["business", "psychology", "science", "nature", "philosophy"]):
        detected.append("Nonfiction")
    
    return detected[:3] if detected else ["Nonfiction"]  # Default to nonfiction


def get_genres_for_titles(titles: List[str], top_k: int) -> List[List[str]]:
    # try subjects first, then batch classify missing
    mapped: List[List[str]] = [map_subjects_to_genres(openlibrary_get_subjects(t), top_k) for t in titles]
    missing_idx = [i for i, m in enumerate(mapped) if not m]
    if not missing_idx:
        return mapped

    clf = get_classifier()
    if clf is None:
        # No AI classifier - use simple fallback
        for idx in missing_idx:
            mapped[idx] = simple_genre_detection(titles[idx])
        return mapped

    batch_size = 16
    still_missing = []
    for i in range(0, len(missing_idx), batch_size):
        batch_idx = missing_idx[i:i+batch_size]
        batch_titles = [titles[j] for j in batch_idx]
        try:
            res = clf(batch_titles, candidate_labels=GENRES, hypothesis_template="This book is a {} book.")
        except Exception:
            res = []
        if isinstance(res, dict):
            res = [res]
        for j, r in enumerate(res):
            labels = r.get("labels", [])[:top_k]
            if labels:
                mapped[batch_idx[j]] = labels
            else:
                still_missing.append(batch_idx[j])
    
    # Final fallback for anything still missing
    for idx in still_missing:
        mapped[idx] = simple_genre_detection(titles[idx])
    
    return mapped


# =========================================================
# CORE PIPELINE
# =========================================================
def run_pipeline(df: pd.DataFrame, output_file: str):
    titles = df["Title"].astype(str).tolist()
    numbers = df["Number"].astype(float).tolist()

    # fast language detection
    langs = [detect_language(t)[1] for t in titles]

    # parallel OpenLibrary lookups
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        subjects_list = list(ex.map(openlibrary_get_subjects, titles))

    # map subjects to genres
    mapped = [map_subjects_to_genres(s, TOP_K_GENRES) for s in subjects_list]

    # batch-classify missing
    missing = [i for i, m in enumerate(mapped) if not m]
    if missing:
        to_classify = [titles[i] for i in missing]
        classified = get_genres_for_titles(to_classify, TOP_K_GENRES)
        for idx, labels in zip(missing, classified):
            mapped[idx] = labels

    rows = []
    genre_title_count = defaultdict(int)
    genre_number_sum = defaultdict(float)

    for title, num, lang, genres in zip(titles, numbers, langs, mapped):
        genres = genres or []
        for g in genres:
            genre_title_count[g] += 1
            genre_number_sum[g] += num / max(len(genres), 1)
        rows.append({"Title": title, "Number": num, "Language": lang, "Genres": ", ".join(genres)})

    result_df = pd.DataFrame(rows)

    top_titles = sorted(genre_title_count.items(), key=lambda x: x[1], reverse=True)[:5]
    top_numbers = sorted(genre_number_sum.items(), key=lambda x: x[1], reverse=True)[:5]
    summary_text = (f"Top genres by title count: {top_titles}. "
                    f"Top genres by total Number: {[(g, int(v)) for g, v in top_numbers]}.")

    output_data = rows + [{"Title": "", "Number": "", "Language": "", "Genres": ""},
                         {"Title": "SUMMARY", "Number": int(result_df["Number"].sum()), "Language": "", "Genres": summary_text}]

    final_df = pd.DataFrame(output_data)
    final_df.to_csv(output_file, index=False, encoding="utf-8")

    _save_cache()
    print(f" Saved {output_file} ({len(rows)} titles processed)")


# =========================================================
# DATA PREPARATION
# =========================================================
def prepare_company_a(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig", on_bad_lines="skip",
                     dtype={"Title": "string", "Number": "string"})
    df["Title"] = df["Title"].astype(str).str.strip()
    df["Number"] = (df["Number"].astype(str).str.strip().str.replace(",", "", regex=False))
    df["Number"] = pd.to_numeric(df["Number"], errors="coerce").fillna(0)
    return df.groupby("Title", as_index=False)["Number"].sum()


# =========================================================
# MAIN
# =========================================================

def main():
    df_in = prepare_company_a(COMPANY_A_FILE)
    run_pipeline(df_in, OUTPUT_COMPANY_A)


if __name__ == "__main__":
    main()


 Saved company_a_genres_output.csv (1082 titles processed)


In [6]:
# DEBUG: Test the pipeline with sample titles
print("=" * 80)
print("DEBUG: Testing genre classification pipeline")
print("=" * 80)

# Load sample data
df_test = prepare_company_a(COMPANY_A_FILE)
print(f"\nLoaded {len(df_test)} unique titles")

# Test with first 5 titles
test_titles = df_test["Title"].head(5).tolist()
print(f"\nTesting with first 5 titles:")
for i, t in enumerate(test_titles, 1):
    print(f"  {i}. {t}")

print("\n--- OpenLibrary Subjects Lookup ---")
for title in test_titles:
    subjects = openlibrary_get_subjects(title)
    print(f"{title}: {subjects[:3] if subjects else 'NO SUBJECTS'}")

print("\n--- Genre Mapping ---")
for title in test_titles:
    subjects = openlibrary_get_subjects(title)
    genres = map_subjects_to_genres(subjects, TOP_K_GENRES)
    print(f"{title}: {genres if genres else 'NO GENRES MAPPED'}")

print("\n--- AI Classifier Test ---")
clf = get_classifier()
if clf is None:
    print("Warning: AI Classifier failed to load")
else:
    print("AI Classifier loaded successfully")
    # Test with titles that had no OpenLibrary genres
    no_genre_titles = [t for t in test_titles if not map_subjects_to_genres(openlibrary_get_subjects(t), TOP_K_GENRES)]
    if no_genre_titles:
        print(f"Testing AI classifier on {len(no_genre_titles)} titles without OpenLibrary genres:")
        try:
            result = clf(no_genre_titles[:2], candidate_labels=GENRES, hypothesis_template="This book is a {} book.")
            print(f"Result: {result}")
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {str(e)[:100]}")

print("\n" + "=" * 80)

DEBUG: Testing genre classification pipeline

Loaded 1082 unique titles

Testing with first 5 titles:
  1. 10 րոպե 38 վայրկյան այս տարօրինակ աշխարհում, հատ
  2. 100 աստվածաշնչյան պատմություն, հատ
  3. 1984, հատ
  4. 1988-ի շարժման 30 ամյակը, հատ
  5. 1րոպե անց, հատ

--- OpenLibrary Subjects Lookup ---
10 րոպե 38 վայրկյան այս տարօրինակ աշխարհում, հատ: NO SUBJECTS
100 աստվածաշնչյան պատմություն, հատ: NO SUBJECTS
1984, հատ: NO SUBJECTS
1988-ի շարժման 30 ամյակը, հատ: NO SUBJECTS
1րոպե անց, հատ: NO SUBJECTS

--- Genre Mapping ---
10 րոպե 38 վայրկյան այս տարօրինակ աշխարհում, հատ: NO GENRES MAPPED
100 աստվածաշնչյան պատմություն, հատ: NO GENRES MAPPED
1984, հատ: NO GENRES MAPPED
1988-ի շարժման 30 ամյակը, հատ: NO GENRES MAPPED
1րոպե անց, հատ: NO GENRES MAPPED

--- AI Classifier Test ---

